# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR<sup>2</sup> dataset using the `mlcroissant` library.

### Dataset Source
This dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples. It includes various fields such as accession, description, coverage percentage, peptide counts, and molecular weight (MW), alongside calculated parameters such as pI and normalized abundances across different samples.

- Croissant schema: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}\n\nDescription: {metadata.description}\n\nVersion: {getattr(metadata, 'version', None)}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List available record sets
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
else:
    # For Croissant 1.0, record sets may be found under `recordSet`.
    record_sets = getattr(metadata, 'recordSet', [])

if not record_sets:
    raise RuntimeError("No record sets found in the metadata.")

for i, record_set in enumerate(record_sets):
    print(f"[{i}] RecordSet @id: {record_set['@id']} | Name: {record_set.get('name', '')}")
    # Print available fields (by @id)
    fields = record_set.get('field', [])
    if fields:
        print("    Fields:")
        for field in fields:
            if isinstance(field, dict):
                print(f"        - {field.get('@id', '')} ({field.get('name', 'No name')})")
            else:
                print(f"        - {field}")
    else:
        print("    No fields listed in this RecordSet.")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Identify record set IDs (using @id)
record_set_ids = [rs['@id'] for rs in record_sets]
print('Available record set @ids:', record_set_ids)

# We'll use the first record set for demonstration
selected_record_set_id = record_set_ids[0]  # Replace with appropriate @id after checking above

# Load records into DataFrame
records = list(dataset.records(record_set=selected_record_set_id))
df = pd.DataFrame(records)

print(f"Columns in the selected record set ({selected_record_set_id}):\n", df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. For demonstration, we'll select a numeric column if available, and perform operations as outlined in the template.

In [ ]:
# Identify numeric columns
numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
print('Numeric fields:', numeric_candidates)

if numeric_candidates:
    # Choose the first numeric field for demonstration
    numeric_field_id = numeric_candidates[0]

    # Filter where numeric_field > threshold (e.g., 10)
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()

    print(f"Filtered records where {numeric_field_id} > {threshold} (showing up to 5 records):")
    print(filtered_df.head())

    # Normalize selected numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized values for '{numeric_field_id}' (mean=0, std=1):")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # If there is a likely group/categorical field, group by it (e.g., 'description' or 'accession')
    group_candidates = [col for col in df.columns if col not in numeric_candidates]
    group_field = None
    if group_candidates:
        group_field = group_candidates[0]
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped and averaged by '{group_field}':\n", grouped_df.head())
else:
    print('No numeric fields available for EDA demonstration.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Plot histogram if numeric field exists
if numeric_candidates:
    plt.figure(figsize=(8,4))
    df[numeric_field_id].hist(bins=30)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group (if group_field selected)
    if group_field:
        df.boxplot(column=numeric_field_id, by=group_field, vert=False, figsize=(10, 6))
        plt.title(f'{numeric_field_id} by {group_field}')
        plt.suptitle('')
        plt.xlabel(numeric_field_id)
        plt.ylabel(group_field)
        plt.show()

## 6. Conclusion
- We demonstrated step-by-step loading, inspection, and analysis of the FAIR^2 dataset using `mlcroissant` and the Croissant schema definition.
- The dataset provides protein-related measurements, enabling filtering, normalization, and groups-based aggregation.
- We visualized both the distribution and grouped statistics for selected attributes.

Consider using more advanced processing and exploring the complete schema for deeper insights into the dataset.